# Week 2 — Data dictionary, coding, and provenance

### Learning goals
- Build a **data dictionary** (column meaning, type, codes)
- Recode categorical variables into readable labels
- Identify what 'missing' could mean and how it affects conclusions

### Deliverable
- Completed data dictionary table
- Short provenance paragraph + 2 caveats


In [ ]:
# Core imports (kept minimal for beginners)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# Dataset URL (UCI Heart Disease - Cleveland)
UCI_URL = "https://www.archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

# Column names for processed.cleveland.data (14 columns commonly used in teaching)
COLS = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach",
        "exang","oldpeak","slope","ca","thal","num"]


In [ ]:
import ssl
import io
import urllib.request # Added this import

def load_raw():
    # Create an unverified SSL context to bypass certificate verification
    ctx = ssl.create_default_context()
    ctx.check_hostname = False
    ctx.verify_mode = ssl.CERT_NONE

    # Open the URL with the unverified context
    with urllib.request.urlopen(UCI_URL, context=ctx) as url_response:
        # Read the content and decode it
        s = url_response.read().decode('utf-8')

    # Use io.StringIO to make the string behave like a file for pandas.read_csv
    df_raw = pd.read_csv(io.StringIO(s), header=None, names=COLS)
    return df_raw

def coerce_types(df_raw):
    # Missing values are sometimes encoded as "?"
    df = df_raw.replace("?", np.nan).copy()

    # Convert numeric-looking columns
    numeric_cols = ["age","trestbps","chol","thalach","oldpeak","ca","thal","num"]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # Binary target: disease present if num > 0 (UCI convention)
    df["disease"] = (df["num"] > 0).astype("Int64")
    return df

df_raw = load_raw()
df = coerce_types(df_raw)

df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,disease
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0,0


In [ ]:
def quick_profile(df):
    """Small, beginner-friendly dataframe profile."""
    display(df.head())
    print("\nShape:", df.shape)
    print("\nDtypes:")
    print(df.dtypes)
    print("\nMissingness (top 10):")
    miss = df.isna().mean().sort_values(ascending=False)
    display(miss.head(10).to_frame("missing_fraction"))
    print("\nBasic describe (numeric):")
    display(df.describe(include=[np.number]).T)

def check_columns(df):
    assert df.columns.is_unique, "Duplicate column names found"
    assert df.shape[0] > 0, "No rows loaded"
    return True


In [ ]:
quick_profile(df)

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num,disease
0,63.0,1.0,1.0,145.0,233.0,1.0,2.0,150.0,0.0,2.3,3.0,0.0,6.0,0,0
1,67.0,1.0,4.0,160.0,286.0,0.0,2.0,108.0,1.0,1.5,2.0,3.0,3.0,2,1
2,67.0,1.0,4.0,120.0,229.0,0.0,2.0,129.0,1.0,2.6,2.0,2.0,7.0,1,1
3,37.0,1.0,3.0,130.0,250.0,0.0,0.0,187.0,0.0,3.5,3.0,0.0,3.0,0,0
4,41.0,0.0,2.0,130.0,204.0,0.0,2.0,172.0,0.0,1.4,1.0,0.0,3.0,0,0



Shape: (303, 15)

Dtypes:
age         float64
sex         float64
cp          float64
trestbps    float64
chol        float64
fbs         float64
restecg     float64
thalach     float64
exang       float64
oldpeak     float64
slope       float64
ca          float64
thal        float64
num           int64
disease       Int64
dtype: object

Missingness (top 10):


,missing_fraction
ca,0.013201
thal,0.006601
age,0.000000
trestbps,0.000000
chol,0.000000
sex,0.000000
cp,0.000000
restecg,0.000000
fbs,0.000000
thalach,0.000000



Basic describe (numeric):


,count,mean,std,min,25%,50%,75%,max
age,303.0,54.438944,9.038662,29.0,48.0,56.0,61.0,77.0
sex,303.0,0.679868,0.467299,0.0,0.0,1.0,1.0,1.0
cp,303.0,3.158416,0.960126,1.0,3.0,3.0,4.0,4.0
trestbps,303.0,131.689769,17.599748,94.0,120.0,130.0,140.0,200.0
chol,303.0,246.693069,51.776918,126.0,211.0,241.0,275.0,564.0
fbs,303.0,0.148515,0.356198,0.0,0.0,0.0,0.0,1.0
restecg,303.0,0.990099,0.994971,0.0,0.0,1.0,2.0,2.0
thalach,303.0,149.607261,22.875003,71.0,133.5,153.0,166.0,202.0
exang,303.0,0.326733,0.469794,0.0,0.0,0.0,1.0,1.0
oldpeak,303.0,1.039604,1.161075,0.0,0.0,0.8,1.6,6.2


## Step 1 — Data dictionary template (fill this table)
Complete: `meaning`, `type`, `codes_or_units`, `notes`.


In [ ]:
data_dict = pd.DataFrame({
    "column": COLS + ["disease"],
    "meaning": [
        "Age of the patient",
        "Biological sex of the patient",
        "Chest pain type",
        "Resting blood pressure at hospital admission",
        "Serum cholesterol level",
        "Fasting blood sugar greater than 120 mg/dL",
        "Resting electrocardiographic results",
        "Maximum heart rate achieved during exercise",
        "Exercise-induced angina",
        "ST depression induced by exercise relative to rest",
        "Slope of the peak exercise ST segment",
        "Number of major vessels colored by fluoroscopy",
        "Thalassemia type (blood disorder affecting hemoglobin)",
        "Angiographic diagnosis of heart disease (original target, 0–4 severity)",
        "Binary heart disease label derived from num (1 = disease present)"
    ],
    "type": [
        "numeric",
        "binary",
        "categorical",
        "numeric",
        "numeric",
        "binary",
        "categorical",
        "numeric",
        "binary",
        "numeric",
        "categorical",
        "numeric",
        "categorical",
        "numeric",
        "binary"
    ],
    "codes_or_units": [
        "years",
        "0 = female, 1 = male",
        "1 = typical angina, 2 = atypical angina, 3 = non-anginal pain, 4 = asymptomatic",
        "mmHg",
        "mg/dL",
        "0 = false (≤120 mg/dL), 1 = true (>120 mg/dL)",
        "0 = normal, 1 = ST-T wave abnormality, 2 = left ventricular hypertrophy (Estes criteria)",
        "bpm",
        "0 = no, 1 = yes",
        "mm (continuous; 0 = no depression)",
        "1 = upsloping, 2 = flat, 3 = downsloping",
        "0–3 (integer); missing values encoded as '?' → NaN after coercion",
        "3 = normal, 6 = fixed defect, 7 = reversible defect; missing values encoded as '?' → NaN",
        "0 = no disease, 1–4 = increasing severity of stenosis",
        "0 = no disease, 1 = disease present (num > 0)"
    ],
    "notes": [
        "No missing values; range ~29–77 years in Cleveland cohort",
        "No missing values; dataset skewed toward male patients (~68%)",
        "No missing values; 'asymptomatic' (4) is counterintuitively the highest-risk group",
        "No missing values; measure taken at rest on hospital admission",
        "No missing values; extreme outliers present (>500 mg/dL)",
        "No missing values; only 15% of patients are true positive",
        "No missing values; majority class is 'normal' (0)",
        "No missing values; lower max HR in older or more diseased patients",
        "No missing values; positively correlated with disease presence",
        "No missing values; continuous variable; value of 0 is clinically normal",
        "No missing values; downsloping (3) associated with ischemia",
        "6 missing values; higher ca values associated with more severe disease",
        "2 missing values; reversible defect most associated with disease",
        "Original multi-class target; most analyses binarize into disease/no disease",
        "Derived binary target; 0 = healthy (num=0), 1 = disease (num=1,2,3,4)"
    ]
})
data_dict

,column,meaning,type,codes_or_units,notes
0,age,Age of the patient,numeric,years,No missing values; range ~29–77 years in Cleve...
1,sex,Biological sex of the patient,binary,"0 = female, 1 = male",No missing values; dataset skewed toward male ...
2,cp,Chest pain type,categorical,"1 = typical angina, 2 = atypical angina, 3 = n...",No missing values; 'asymptomatic' (4) is count...
3,trestbps,Resting blood pressure at hospital admission,numeric,mmHg,No missing values; measure taken at rest on ho...
4,chol,Serum cholesterol level,numeric,mg/dL,No missing values; extreme outliers present (>...
5,fbs,Fasting blood sugar greater than 120 mg/dL,binary,"0 = false (≤120 mg/dL), 1 = true (>120 mg/dL)",No missing values; only 15% of patients are tr...
6,restecg,Resting electrocardiographic results,categorical,"0 = normal, 1 = ST-T wave abnormality, 2 = lef...",No missing values; majority class is 'normal' (0)
7,thalach,Maximum heart rate achieved during exercise,numeric,bpm,No missing values; lower max HR in older or mo...
8,exang,Exercise-induced angina,binary,"0 = no, 1 = yes",No missing values; positively correlated with ...
9,oldpeak,ST depression induced by exercise relative to ...,numeric,mm (continuous; 0 = no depression),No missing values; continuous variable; value ...


## Step 2 — Inspect coded categoricals


In [ ]:
coded_cols = ["sex","cp","fbs","restecg","exang","slope","ca","thal","disease"]
for c in coded_cols:
    print("\n", c)
    display(df[c].value_counts(dropna=False).sort_index())



 sex


,count
sex,
0.0,97
1.0,206



 cp


,count
cp,
1.0,23
2.0,50
3.0,86
4.0,144



 fbs


...

,count
ca,
0.0,176
1.0,65
2.0,38
3.0,20
NaN,4



 thal


,count
thal,
3.0,166
6.0,18
7.0,117
NaN,2



 disease


,count
disease,
0,164
1,139


## Step 3 — Create human-readable labels (starter mappings)
Treat these as assumptions unless you verify from documentation.


In [ ]:
sex_map = {0: "female", 1: "male"}
exang_map = {0: "no", 1: "yes"}
fbs_map = {0: "false", 1: "true"}  # fasting blood sugar > 120 mg/dl
cp_map = {1: "typical angina", 2: "atypical angina", 3: "non-anginal pain", 4: "asymptomatic"}

df_labeled = df.copy()
df_labeled["sex_label"] = df_labeled["sex"].map(sex_map)
df_labeled["exang_label"] = df_labeled["exang"].map(exang_map)
df_labeled["fbs_label"] = df_labeled["fbs"].map(fbs_map)
df_labeled["cp_label"] = df_labeled["cp"].map(cp_map)

df_labeled[["sex","sex_label","cp","cp_label","exang","exang_label","fbs","fbs_label"]].head(10)


,sex,sex_label,cp,cp_label,exang,exang_label,fbs,fbs_label
0,1.0,male,1.0,typical angina,0.0,no,1.0,true
1,1.0,male,4.0,asymptomatic,1.0,yes,0.0,false
2,1.0,male,4.0,asymptomatic,1.0,yes,0.0,false
3,1.0,male,3.0,non-anginal pain,0.0,no,0.0,false
4,0.0,female,2.0,atypical angina,0.0,no,0.0,false
5,1.0,male,2.0,atypical angina,0.0,no,0.0,false
6,0.0,female,4.0,asymptomatic,0.0,no,0.0,false
7,0.0,female,4.0,asymptomatic,1.0,yes,0.0,false
8,1.0,male,4.0,asymptomatic,0.0,no,0.0,false
9,1.0,male,4.0,asymptomatic,1.0,yes,1.0,true


## TODO — Provenance & caveats
Write:
- Source: UCI Machine Learning Repository — Heart Disease Dataset (Cleveland Clinic Foundation).Collected by Robert Detrano, M.D., Ph.D., V.A. Medical Center, Long Beach, CA.
Original data: 1988. URL: https://archive.ics.uci.edu/ml/datasets/Heart+Disease

- Likely population: Adult patients (age ~29–77) who presented at the Cleveland Clinic with symptoms warranting coronary angiography. This is a clinical  referral population, not a random sample of the general public.

- Caveat 1: Patients in this dataset were already referred for
angiography due to suspected heart disease. This means the dataset over-represents symptomatic individuals compared to the general population. A model trained on this data should not be used to screen healthy adults.

- Caveat 2: The `ca` (6 missing) and `thal` (2 missing) columns contain values that were originally encoded as "?" in the raw file. It is unclear whether these values were not measured, lost during data entry, or genuinely unknown. Dropping vs. imputing these rows could meaningfully affect model performance and conclusions.


